# Generalizability Evaluation: Function Vectors in Large Language Models

## Overview
This notebook evaluates the generalizability of the findings from the Function Vectors research (Todd et al., ICLR 2024).

### Evaluation Checklist:
- **GT1**: Generalization to a New Model
- **GT2**: Generalization to New Data
- **GT3**: Method/Specificity Generalizability

### Key Findings from Original Research:
- Function vectors are compact representations of input-output functions encoded in attention heads
- They cluster in middle layers (~L/3 depth)
- Models used in original work: GPT-J, GPT-NeoX, Llama 2 (7B, 13B, 70B), GPT-2 XL

In [1]:
# Setup environment and working directory
import os
os.chdir('/home/smallyan/eval_agent')

# Load bashrc for environment variables
bashrc_path = os.path.expanduser('~/.bashrc')
with open(bashrc_path) as f:
    for line in f:
        line = line.strip()
        if line.startswith('export ') and '=' in line:
            var_def = line[7:]  # Remove 'export '
            if '=' in var_def:
                key, value = var_def.split('=', 1)
                value = value.strip('"').strip("'")
                os.environ[key] = value

print("HF_HOME:", os.environ.get('HF_HOME', 'Not set'))
print("CUDA available:", end=" ")

import torch
print(torch.cuda.is_available())
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

HF_HOME: /net/projects2/chai-lab/shared_models
CUDA available: 

True
GPU: NVIDIA H100 NVL
GPU Memory: 99.95 GB


In [2]:
# Check what models are cached in the HF_HOME hub
import os
hub_path = "/net/projects2/chai-lab/shared_models/hub"
print("Cached models in HF_HOME:")
if os.path.exists(hub_path):
    models_dir = os.path.join(hub_path, "models--*")
    import glob
    cached = glob.glob(os.path.join(hub_path, "models--*"))
    for model in sorted(cached):
        print(f"  - {os.path.basename(model)}")
else:
    print("Hub directory not found")

Cached models in HF_HOME:
  - models--BAAI--bge-base-en-v1.5
  - models--EleutherAI--gpt-j-6B
  - models--EleutherAI--gpt-j-6b
  - models--EleutherAI--gpt-neo-1.3B
  - models--EleutherAI--gpt-neo-125M
  - models--EleutherAI--pythia-1.4b
  - models--EleutherAI--pythia-2.8b
  - models--EleutherAI--pythia-410m
  - models--EleutherAI--pythia-6.9b
  - models--HuggingFaceH4--zephyr-7b-beta
  - models--NeelNanda--Attn_Only_2L512W_C4_Code
  - models--NeelNanda--Attn_Only_4L512W_C4_Code
  - models--NeelNanda--gpt-neox-tokenizer-digits
  - models--Qwen--Qwen2-0.5B
  - models--Qwen--Qwen2.5-0.5B
  - models--Qwen--Qwen2.5-0.5B-Instruct
  - models--Qwen--Qwen2.5-1.5B
  - models--Qwen--Qwen2.5-32B-Instruct
  - models--Qwen--Qwen2.5-7B-Instruct
  - models--Qwen--Qwen3-0.6B
  - models--Qwen--Qwen3-32B
  - models--baulab--elm-zephyr-7b-beta
  - models--bert-base-uncased
  - models--google--gemma-2-27b-it
  - models--google--gemma-2-2b
  - models--google--gemma-2-2b-it
  - models--google--gemma-2-9b-it


In [3]:
# Add the function vectors repo to path
import sys
sys.path.insert(0, '/net/scratch2/smallyan/function_vectors_eval/src')

# Import the necessary utilities
from utils.extract_utils import get_mean_head_activations, compute_function_vector
from utils.intervention_utils import fv_intervention_natural_text
from utils.prompt_utils import load_dataset
from utils.model_utils import load_gpt_model_and_tokenizer

print("Imports successful!")

RuntimeError: Failed to find C compiler. Please specify via CC environment variable.

In [4]:
# Set CC environment variable and retry
import os
os.environ['CC'] = '/usr/bin/gcc'

# Reload bitsandbytes related modules
import importlib
import sys

# Remove the cached import attempts
mods_to_remove = [m for m in sys.modules if 'bitsandbytes' in m or 'triton' in m]
for mod in mods_to_remove:
    del sys.modules[mod]

# Now try importing again
from utils.extract_utils import get_mean_head_activations, compute_function_vector
from utils.intervention_utils import fv_intervention_natural_text
from utils.prompt_utils import load_dataset
from utils.model_utils import load_gpt_model_and_tokenizer

print("Imports successful!")

gcc: fatal error: cannot execute ‘as’: execvp: No such file or directory
compilation terminated.


CalledProcessError: Command '['/usr/bin/gcc', '/tmp/tmptekx9l90/main.c', '-O3', '-shared', '-fPIC', '-Wno-psabi', '-o', '/tmp/tmptekx9l90/cuda_utils.cpython-311-x86_64-linux-gnu.so', '-lcuda', '-L/home/smallyan/.conda/envs/meta/lib/python3.11/site-packages/triton/backends/nvidia/lib', '-L/lib/x86_64-linux-gnu', '-I/home/smallyan/.conda/envs/meta/lib/python3.11/site-packages/triton/backends/nvidia/include', '-I/tmp/tmptekx9l90', '-I/home/smallyan/.conda/envs/meta/include/python3.11']' returned non-zero exit status 1.

In [5]:
# Let me implement the function vector utilities manually without bitsandbytes
# This will allow us to work with models that don't require quantization

import torch
import json
import os
from baukit import TraceDict

def get_module(model, name):
    """Finds the named module within the given model."""
    for n, m in model.named_modules():
        if n == name:
            return m
    raise LookupError(name)

def add_function_vector(edit_layer, fv_vector, device, idx=-1):
    """
    Adds a vector to the output of a specified layer in the model.
    """
    def add_act(output, layer_name):
        current_layer = int(layer_name.split(".")[2])
        if current_layer == edit_layer:
            if isinstance(output, tuple):
                output[0][:, idx] += fv_vector.to(device)
                return output
            else:
                return output
        else:
            return output
    return add_act

def gather_attn_activations(model_config, model, tokenizer, prompts, device):
    """
    Gathers attention head activations for a given set of prompts.
    """
    n_layers = model_config['n_layers']
    n_heads = model_config['n_heads']
    head_dim = model_config['resid_dim'] // n_heads
    
    all_activations = []
    
    for prompt in prompts:
        inputs = tokenizer(prompt, return_tensors='pt').to(device)
        
        # Hook to extract attention head activations
        activations = {}
        def make_hook(layer_name):
            def hook(module, input, output):
                if isinstance(input, tuple):
                    input = input[0]
                activations[layer_name] = input.detach()
            return hook
        
        hooks = []
        for layer in range(n_layers):
            layer_name = model_config['attn_hook_names'][layer]
            module = get_module(model, layer_name)
            hooks.append(module.register_forward_hook(make_hook(layer_name)))
        
        with torch.no_grad():
            model(**inputs)
        
        # Remove hooks
        for hook in hooks:
            hook.remove()
        
        # Collect activations - reshape to (n_layers, n_heads, n_tokens, head_dim)
        prompt_acts = []
        for layer in range(n_layers):
            layer_name = model_config['attn_hook_names'][layer]
            act = activations[layer_name]  # (batch, tokens, hidden_dim)
            act = act.view(act.shape[0], act.shape[1], n_heads, head_dim)  # (batch, tokens, heads, head_dim)
            act = act.permute(0, 2, 1, 3).squeeze(0)  # (heads, tokens, head_dim)
            prompt_acts.append(act)
        
        prompt_acts = torch.stack(prompt_acts, dim=0)  # (n_layers, n_heads, n_tokens, head_dim)
        all_activations.append(prompt_acts)
    
    return all_activations

print("Function vector utilities defined successfully!")

Function vector utilities defined successfully!


## GT1: Generalization to a New Model

**Original Models Used:** GPT-J (6B), GPT-NeoX (20B), Llama 2 (7B, 13B, 70B), GPT-2 XL

**Test Model:** Pythia-2.8B (EleutherAI) - NOT used in the original paper

The function vectors research claims that:
1. Function vectors are encoded in attention heads
2. They cluster in middle layers (~L/3)
3. Adding the function vector enables zero-shot task execution

We will test if these findings generalize to Pythia-2.8B.

In [6]:
# Define the model loading function directly (avoiding the import chain that requires bitsandbytes)
from transformers import AutoModelForCausalLM, AutoTokenizer

def load_pythia_model(model_name, device='cuda'):
    """Load Pythia model with configuration."""
    print(f"Loading: {model_name}")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16).to(device)
    
    MODEL_CONFIG = {
        "n_heads": model.config.num_attention_heads,
        "n_layers": model.config.num_hidden_layers,
        "resid_dim": model.config.hidden_size,
        "name_or_path": model.config.name_or_path,
        "attn_hook_names": [f'gpt_neox.layers.{layer}.attention.dense' for layer in range(model.config.num_hidden_layers)],
        "layer_hook_names": [f'gpt_neox.layers.{layer}' for layer in range(model.config.num_hidden_layers)],
        "prepend_bos": False
    }
    
    return model, tokenizer, MODEL_CONFIG

# Load the Pythia-2.8B model (NOT used in the original paper)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model, tokenizer, model_config = load_pythia_model("EleutherAI/pythia-2.8b", device=device)

print(f"\nModel Configuration:")
print(f"  Layers: {model_config['n_layers']}")
print(f"  Heads: {model_config['n_heads']}")
print(f"  Hidden Dim: {model_config['resid_dim']}")
print(f"  Device: {model.device}")

/home/smallyan/.conda/envs/meta/lib/python3.11/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


Loading: EleutherAI/pythia-2.8b


`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/5.68G [00:00<?, ?B/s]

In [7]:
# Wait for model to finish loading if still in progress
print(f"\nModel Configuration:")
print(f"  Layers: {model_config['n_layers']}")
print(f"  Heads: {model_config['n_heads']}")
print(f"  Hidden Dim: {model_config['resid_dim']}")
print(f"  Device: {model.device}")

In [8]:
# Check if the model is loaded
print(f"Model type: {type(model)}")
print(f"Model device: {next(model.parameters()).device}")

In [9]:
import time
time.sleep(2)
# Verify model is loaded
print(f"Model type: {type(model)}")
print(f"Model device: {next(model.parameters()).device}")
print(f"Model config n_layers: {model_config['n_layers']}")

In [10]:
# Re-check
print("Checking model loaded...")
print(model_config)